# SEC Filings Pipeline

Full pipeline from scraping 10-K filings through BigQuery AI extraction and property graph creation.

## Pipeline Steps
1. **Google Cloud authentication** – gcloud auth
2. **Scraper** – Download 10-K filings from SEC.gov
3. **Parser** – Parse SGML to Markdown (optional, for viewing)
4. **Extract Sections** – Parse SGML to JSONL (Item 1, 1A, 7; feeds BigQuery)
5. **Upload JSONL to GCS** – Sync JSON to BigQuery load bucket
6. **Init Tables** – Create sections, insights, staging tables
7. **Extraction** – AI extraction (Gemini) → insights
8. **Create Graph** – Transform insights to graph_edges
9. **Prepare Property Graph** – Node and edge tables
10. **Add Action to Market** – market_action column
11. **Create Property Graph DDL** – SecGraph property graph

## 0. Colab Setup
If you are running this in Google Colab, you need to clone the repository and install requirements to access the custom scraper scripts and SQL files.

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    from IPython import get_ipython
    ipy = get_ipython()
    if not os.path.exists('00.0_scraper.py'):
        ipy.system('git clone https://github.com/Kineviz/fortune500.git')
        os.chdir('fortune500')
    ipy.system('pip install -r requirements.txt')


## 1. Setup & Google Cloud Authentication

In [ ]:
# Set GCP_PROJECT (replace YOUR_PROJECT_ID); skip auth if not set
GCP_PROJECT = "YOUR_PROJECT_ID"  # e.g. "kineviz-bigquery-graph"

if GCP_PROJECT != "YOUR_PROJECT_ID":
    from IPython import get_ipython
    ipy = get_ipython()
    ipy.system(f"gcloud config set project {GCP_PROJECT}")
    ipy.system("gcloud auth login")
    result = __import__("subprocess").run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True, text=True
    )
    current = (result.stdout or "").strip() or "(unset)"
    print(f"✓ Project verified: {current}") if current == GCP_PROJECT else print(f"⚠ Current: {current}")
else:
    raise ValueError("Set GCP_PROJECT above (replace YOUR_PROJECT_ID), then re-run this cell.")

In [ ]:
# Ensure we're in the project root (where 00.0_scraper.py, SQL files, list.csv live)
import os

def find_project_root():
    cwd = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(cwd, "00.0_scraper.py")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return os.getcwd()

root = find_project_root()
os.chdir(root)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Configure these for your environment
import os

GCS_BUCKET = "gs://kineviz_fortune500_sec_filing"  # Your GCS bucket
BQ_PROJECT = globals().get("GCP_PROJECT", "YOUR_PROJECT_ID")  # Copy from GCP_PROJECT (set in auth cell)
BQ_LOCATION = "US"
SGML_DIR = "data/sgml"
MARKDOWN_DIR = "data/markdown"
JSON_DIR = "data/json"

# Pipeline options
SCRAPER_LIMIT = 5        # Companies from list.csv (use small number for testing)
SCRAPER_LAST_N_YEARS = 2  # Years of filings to fetch
SCRAPER_OUTPUT = SGML_DIR

In [ ]:
# Verify GCS bucket exists and is accessible
import subprocess

result = subprocess.run(
    ["gsutil", "ls", GCS_BUCKET],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print(f"✓ Bucket accessible: {GCS_BUCKET}")
else:
    print(f"Bucket not found. Creating {GCS_BUCKET}...")
    create = subprocess.run(["gsutil", "mb", "-l", BQ_LOCATION, GCS_BUCKET], capture_output=True, text=True)
    if create.returncode == 0:
        print(f"✓ Bucket created: {GCS_BUCKET}")
    else:
        print(f"⚠ Failed to create bucket: {create.stderr or create.stdout}")

## 2. Run Scraper (00.0_scraper.py)

In [ ]:
import asyncio
import sys

# Add project root to path
sys.path.insert(0, os.getcwd())

from importlib.util import spec_from_file_location, module_from_spec

# Load scraper module
spec = spec_from_file_location("scraper", "00.0_scraper.py")
scraper_mod = module_from_spec(spec)
spec.loader.exec_module(scraper_mod)

scraper = scraper_mod.SECScraper(
    limit=SCRAPER_LIMIT,
    last_n_years=SCRAPER_LAST_N_YEARS,
    workers=5,
    output_dir=SCRAPER_OUTPUT,
    dry_run=False,
)

from tqdm.auto import tqdm
for _ in tqdm([1], desc="Scraping 10-K filings from SEC.gov"):
    await scraper.run()  # Jupyter has its own event loop; use await instead of asyncio.run()

## 3. Run Parser (00.1_parser.py)

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Run parser as subprocess (reliable in Colab; ProcessPoolExecutor + dynamic imports often fails)
for _ in tqdm([1], desc="Parsing SGML to Markdown"):
    subprocess.run(
        [sys.executable, "00.1_parser.py", "--input_base", SGML_DIR, "--output_base", MARKDOWN_DIR],
        check=True,
        cwd=os.getcwd(),
    )

## 4. Extract Sections (01_extract_sections.py)

In [ ]:
import subprocess
import sys
from tqdm.auto import tqdm

# Run extract_sections as subprocess (reliable in Colab; ProcessPoolExecutor + dynamic imports often fails)
for _ in tqdm([1], desc="Extracting sections to JSONL"):
    subprocess.run(
        [sys.executable, "01_extract_sections.py", "--input_base", SGML_DIR, "--output_base", JSON_DIR],
        check=True,
        cwd=os.getcwd(),
    )

## 5. Upload JSONL to GCS

In [ ]:
import os
from IPython import get_ipython
from tqdm.auto import tqdm
json_src = os.path.abspath(JSON_DIR)
os.makedirs(json_src, exist_ok=True)
if not any(os.scandir(json_src)):
    print(f"⚠ {json_src} is empty. Run Steps 2–4 (scraper, parser, extract sections) first, then re-run this cell.")
else:
    for _ in tqdm([1], desc="Uploading JSONL to GCS"):
        get_ipython().system(f'gsutil -m rsync -r {json_src}/ {GCS_BUCKET}/json')
    print(f"✓ Synced to {GCS_BUCKET}/json")

## 6. BigQuery Pipeline

In [ ]:
import subprocess
import glob
from tqdm.auto import tqdm

SCHEMA = "filing_id:STRING,company:STRING,company_name:STRING,cik:STRING,sic:STRING,irs_number:STRING,state_of_inc:STRING,org_name:STRING,sec_file_number:STRING,film_number:STRING,business_street_1:STRING,business_street_2:STRING,business_city:STRING,business_state:STRING,business_zip:STRING,business_phone:STRING,mail_street_1:STRING,mail_street_2:STRING,mail_city:STRING,mail_state:STRING,mail_zip:STRING,filing_url:STRING,year:INTEGER,section_id:STRING,content:STRING"


def run_bq_query(sql_file: str):
    sql_path = os.path.join(os.getcwd(), sql_file)
    if not os.path.exists(sql_path):
        for d in [os.getcwd()] + [os.path.dirname(os.getcwd())] * 3:
            p = os.path.join(d, sql_file)
            if os.path.exists(p):
                sql_path = p
                break
    with open(sql_path) as f:
        sql = f.read()
    cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}"]
    proj = globals().get("BQ_PROJECT", "YOUR_PROJECT_ID")
    if proj and proj != "YOUR_PROJECT_ID":
        cmd.extend([f"--project_id={proj}"])
    subprocess.run(cmd, input=sql.encode(), check=True)


def run_bq_load(uris: str):
    cmd = ["bq", "load", "--source_format=NEWLINE_DELIMITED_JSON", "--replace", f"--schema={SCHEMA}", "sec_filings.sections_staging", uris]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        cmd.insert(-1, f"--project_id={BQ_PROJECT}")
    subprocess.run(cmd, check=True, capture_output=True)


def process_batch(uris: str):
    run_bq_load(uris)
    run_bq_query("04_extraction.sql")
    insert_cmd = ["bq", "query", "--use_legacy_sql=false", f"--location={BQ_LOCATION}",
         "INSERT INTO sec_filings.sections SELECT * FROM sec_filings.sections_staging"]
    bq_proj = globals().get("BQ_PROJECT", "")
    if bq_proj and bq_proj != "YOUR_PROJECT_ID":
        insert_cmd.insert(-1, f"--project_id={BQ_PROJECT}")
    subprocess.run(insert_cmd, check=True)

In [ ]:
# 6.0 Initialize tables
for _ in tqdm([1], desc="Initializing BigQuery tables"):
    run_bq_query("03_init_tables.sql")
print("Tables initialized.")

In [ ]:
# 6.1 Batch load, extract, archive
from tqdm.auto import tqdm
company_dirs = sorted(glob.glob(os.path.join(JSON_DIR, "*")))
company_dirs = [d for d in company_dirs if os.path.isdir(d)]

gcs_prefix = f"{GCS_BUCKET}/json"
total = len(company_dirs)

for i, company_dir in enumerate(tqdm(company_dirs, desc="Processing companies")):
    sections = glob.glob(os.path.join(company_dir, "*", "sections.jsonl"))
    if not sections:
        continue
    uris = [f"{gcs_prefix}/" + os.path.relpath(p, JSON_DIR).replace(os.sep, "/") for p in sections]
    uris_str = ",".join(uris)
    process_batch(uris_str)

In [ ]:
# 6.2 Create graph edges
for _ in tqdm([1], desc="Creating graph edges"):
    run_bq_query("05_create_graph.sql")
print("Graph edges created.")

In [ ]:
# 6.3 Prepare node/edge tables
for _ in tqdm([1], desc="Preparing node/edge tables"):
    run_bq_query("06_prepare_property_graph.sql")
print("Property graph tables created.")

In [ ]:
# 6.4 Add market_action to nodes_market
# Note: If market_action column already exists, ALTER TABLE will fail on re-run.
# Edit 06_1_add_action_to_market.sql to remove the ADD COLUMN line in that case.
for _ in tqdm([1], desc="Adding market_action to nodes_market"):
    run_bq_query("06_1_add_action_to_market.sql")
print("Market action column added.")

In [ ]:
# 6.5 Create SecGraph property graph
for _ in tqdm([1], desc="Creating SecGraph property graph"):
    run_bq_query("07_create_property_graph_ddl.sql")
print("Pipeline complete.")